<!-- # The Bayesian Finite Element Method in Inverse Problems: Pullout Test

This notebook is associated with section 3.1 of "The Bayesian Finite Element Method in Inverse Problems: a Critical Comparison between Probabilistic Models for Discretization Error" by Anne Poot, Iuri Rocha, Pierre Kerfriden and Frans van der Meer ([doi:10.48550/arXiv.2506.02815](https://doi.org/10.48550/arXiv.2506.02815)). -->
# Convergence

In [ ]:
# general imports
import os
import numpy as np
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
from sksparse.cholmod import cholesky
import matplotlib.pyplot as plt
from warnings import warn

from myjivex.util import QuickViewer

from fem.jive import CJiveRunner
from fem.meshing import (
    mesh_interval_with_line2,
    mesh_rectangle_with_tri3,
    create_unit_mass_matrix,
    create_bboxes,
    list_bbox_bbox_intersections,
    clip_polygons,
)

from experiments.reproduction.theory.props import get_fem_props

## Convergence in 1D

In [ ]:
def true_f_ux(x):
    return (1 - x**2) * (np.exp(2 * x) - 1)


def true_f_uy(y):
    return np.sin(np.pi * y)


def true_f_dux_dx(x):
    return 2 * x + np.exp(2 * x) * (2 - 2 * x - 2 * x**2)


def true_f_duy_dy(y):
    return np.pi * np.cos(np.pi * y)


def true_f_d2ux_dx2(x):
    return 2 + np.exp(2 * x) * (2 - 8 * x - 4 * x**2)


def true_f_d2uy_dy2(y):
    return -np.pi**2 * np.sin(np.pi * y)

In [ ]:
def true_f_solution_1d(coord):
    assert coord.shape[-1] == 1
    x = coord[..., 0]
    return true_f_ux(x)


def true_f_strain_1d(coord):
    assert coord.shape[-1] == 1
    x = coord[..., 0]
    return true_f_dux_dx(x)


def true_f_source_1d(coord):
    assert coord.shape[-1] == 1
    x = coord[..., 0]
    return -true_f_d2ux_dx2(x)


def true_f_solution_2d(coord):
    assert coord.shape[-1] == 2
    x = coord[..., 0]
    y = coord[..., 1]

    ux = true_f_ux(x)
    uy = true_f_uy(y)
    return ux * uy


def true_f_strain_2d(coord):
    assert coord.shape[-1] == 2
    x = coord[..., 0]
    y = coord[..., 1]

    ux = true_f_ux(x)
    uy = true_f_uy(y)
    dux_dx = true_f_dux_dx(x)
    duy_dy = true_f_duy_dy(y)
    return np.array([dux_dx * uy, ux * duy_dy])


def true_f_source_2d(coord):
    assert coord.shape[-1] == 2
    x = coord[..., 0]
    y = coord[..., 1]

    ux = true_f_ux(x)
    uy = true_f_uy(y)
    d2ux_dx2 = true_f_d2ux_dx2(x)
    d2uy_dy2 = true_f_d2uy_dy2(y)
    return -(d2ux_dx2 * uy + ux * d2uy_dy2)

In [ ]:
dimensionality = 2

if dimensionality == 1:

    def true_f_solution(coord):
        return true_f_solution_1d(coord)

    def true_f_strain(coord):
        return true_f_strain_1d(coord)

    def true_f_source(coord):
        return true_f_source_1d(coord)

elif dimensionality == 2:

    def true_f_solution(coord):
        return true_f_solution_2d(coord)

    def true_f_strain(coord):
        return true_f_strain_2d(coord)

    def true_f_source(coord):
        return true_f_source_2d(coord)

else:
    assert False

In [ ]:
# true solution
x = np.linspace(0, 1, 101)
y = np.linspace(0, 1, 101)

if dimensionality == 1:
    domain_coords = x.reshape(-1, 1)
elif dimensionality == 2:
    X, Y = np.meshgrid(x, y)
    domain_coords = np.column_stack([X.ravel(), Y.ravel()])

ndom = len(domain_coords)
u = np.zeros(ndom)
eps = np.zeros((ndom, dimensionality))
f = np.zeros(ndom)

for i, coord in enumerate(domain_coords):
    u[i] = true_f_solution(coord)
    eps[i] = true_f_strain(coord)
    f[i] = true_f_source(coord)

if dimensionality == 1:
    fig, axs = plt.subplots(ncols=3, figsize=(9, 3))
    axs[0].plot(x, u)
    axs[1].plot(x, eps)
    axs[2].plot(x, f)
    plt.show()

elif dimensionality == 2:
    fig, axs = plt.subplots(ncols=4, figsize=(12, 3))
    axs[0].contourf(X, Y, u.reshape(101, 101), levels=20)
    axs[1].contourf(X, Y, eps[..., 0].reshape(101, 101), levels=20)
    axs[2].contourf(X, Y, eps[..., 1].reshape(101, 101), levels=20)
    axs[3].contourf(X, Y, f.reshape(101, 101), levels=20)
    plt.show()

In [ ]:
if dimensionality == 1:
    nodes, elems = mesh_interval_with_line2(n=8)
elif dimensionality == 2:
    nodes, elems = mesh_rectangle_with_tri3(n=256)

props = get_fem_props(dimensionality)
jive = CJiveRunner(props, elems=elems)
globdat = jive()

In [ ]:
if dimensionality == 1:
    x_h = nodes.get_coords().flatten()
    u_h = globdat["state0"]

    fig, ax = plt.subplots()
    ax.plot(x_h, u_h)
    plt.show()

elif dimensionality == 2:
    QuickViewer(globdat["state0"], globdat, comp=0)

In [ ]:
obs_points = np.linspace(0, 1, 16, endpoint=False)
obs_points += 0.5 * (obs_points[1] - obs_points[0])
obs_radius = 0.01

### 1D prior convergence with reference mesh refinement

In [ ]:
def true_g_d2ux_dx2(x, ax, bx):
    if x < ax:
        return 0.0
    elif x > bx:
        return 0.0
    else:
        dx = bx - ax
        return -1 / dx


def true_g_d2uy_dy2(y, ay, by):
    if y < ay:
        return 0.0
    elif y > by:
        return 0.0
    else:
        dy = by - ay
        return -1 / dy


def true_g_dux_dx(x, ax, bx):
    mx = 0.5 * (ax + bx)
    dx = bx - ax
    A = 0.5 - mx
    B = mx * (1 - mx) - dx / 8

    if x < ax:
        return 1 - mx
    elif x > bx:
        return -mx
    else:
        sx = x - mx
        return -sx / dx + A


def true_g_duy_dy(y, ay, by):
    my = 0.5 * (ay + by)
    dy = by - ay
    A = 0.5 - my
    B = my * (1 - my) - dy / 8

    if y < ay:
        return 1 - my
    elif y > by:
        return -my
    else:
        sy = y - my
        return -sy / dy + A


def true_g_ux(x, ax, bx):
    mx = 0.5 * (ax + bx)
    dx = bx - ax
    A = 0.5 - mx
    B = mx * (1 - mx) - dx / 8

    if x < ax:
        return x * (1 - mx)
    elif x > bx:
        return (1 - x) * mx
    else:
        return -(x - ax) * (x - bx) / (2 * dx) + (0.5 - mx) * x + 0.5 * ax


def true_g_uy(y, ay, by):
    my = 0.5 * (ay + by)
    dy = by - ay
    A = 0.5 - my
    B = my * (1 - my) - dy / 8

    if y < ay:
        return y * (1 - my)
    elif y > by:
        return (1 - y) * my
    else:
        return -(y - ay) * (y - by) / (2 * dy) + (0.5 - my) * y + 0.5 * ay

In [ ]:
def true_g_source_1d(coord, a, b):
    assert coord.shape[-1] == 1
    x = coord[..., 0]
    ax, bx = a[0], b[0]
    return -true_g_d2ux_dx2(x, ax, bx)


def true_g_strain_1d(coord, a, b):
    assert coord.shape[-1] == 1
    x = coord[..., 0]
    ax, bx = a[0], b[0]
    return true_g_dux_dx(x, ax, bx)


def true_g_solution_1d(coord, a, b):
    assert coord.shape[-1] == 1
    x = coord[..., 0]
    ax, bx = a[0], b[0]
    return true_g_ux(x, ax, bx)


def true_g_source_2d(coord, a, b):
    assert coord.shape[-1] == 2
    x = coord[..., 0]
    y = coord[..., 1]
    ax, ay = a
    bx, by = b

    ux = true_g_ux(x, ax, bx)
    uy = true_g_uy(y, ay, by)
    d2ux_dx2 = true_g_d2ux_dx2(x, ax, bx)
    d2uy_dy2 = true_g_d2uy_dy2(y, ay, by)

    return -(d2ux_dx2 * uy + ux * d2uy_dy2)


def true_g_strain_2d(coord, a, b):
    assert coord.shape[-1] == 2
    x = coord[..., 0]
    y = coord[..., 1]
    ax, ay = a
    bx, by = b

    ux = true_g_ux(x, ax, bx)
    uy = true_g_uy(y, ay, by)
    dux_dx = true_g_dux_dx(x, ay, by)
    duy_dy = true_g_duy_dy(y, ay, by)
    return np.array([dux_dx * uy, ux * duy_dy])


def true_g_solution_2d(coord, a, b):
    assert coord.shape[-1] == 2
    x = coord[..., 0]
    y = coord[..., 1]
    ax, ay = a
    bx, by = b

    ux = true_g_ux(x, ax, bx)
    uy = true_g_uy(y, ay, by)
    return ux * uy

In [ ]:
if dimensionality == 1:

    def true_g_solution(coord, a, b):
        return true_g_solution_1d(coord, a, b)

    def true_g_strain(coord, a, b):
        return true_g_strain_1d(coord, a, b)

    def true_g_source(coord, a, b):
        return true_g_source_1d(coord, a, b)

elif dimensionality == 2:

    def true_g_solution(coord, a, b):
        return true_g_solution_2d(coord, a, b)

    def true_g_strain(coord, a, b):
        return true_g_strain_2d(coord, a, b)

    def true_g_source(coord, a, b):
        return true_g_source_2d(coord, a, b)

else:
    assert False

In [ ]:
ug = np.zeros(ndom)
epsg = np.zeros((ndom, dimensionality))
g = np.zeros(ndom)

a = (0.49,) * dimensionality
b = (0.51,) * dimensionality

for i, coord in enumerate(domain_coords):
    ug[i] = true_g_solution(coord, a, b)
    epsg[i] = true_g_strain(coord, a, b)
    g[i] = true_g_source(coord, a, b)

if dimensionality == 1:
    fig, axs = plt.subplots(ncols=3, figsize=(9, 3))
    axs[0].plot(x, ug)
    axs[1].plot(x, epsg)
    axs[2].plot(x, g)
    plt.show()

elif dimensionality == 2:
    fig, axs = plt.subplots(ncols=4, figsize=(12, 3))
    axs[0].contourf(X, Y, ug.reshape(101, 101), levels=20)
    axs[1].contourf(X, Y, epsg[..., 0].reshape(101, 101), levels=20)
    axs[2].contourf(X, Y, epsg[..., 1].reshape(101, 101), levels=20)
    axs[3].contourf(X, Y, g.reshape(101, 101), levels=20)
    plt.show()

In [ ]:
def true_g_ux_H0_norm(ax, bx):
    mx = 0.5 * (ax + bx)
    dx = bx - ax
    A = 0.5 - mx
    B = mx * (1 - mx) - dx / 8

    # int_0^a -> 1/3 (1 - m)**2 a**3
    # int_b^1 -> 1/3 m**2 (1 - b)**3
    # int_a^b -> 1/320 d**3 + 1/12 A**2 d**3 - 1/12 B d**2 + B**2 d
    eval_0a = (1 - mx) ** 2 * ax**3 / 3
    eval_ab = dx**3 / 320 + A**2 * dx**3 / 12 - B * dx**2 / 12 + B**2 * dx
    eval_b1 = mx**2 * (1 - bx) ** 3 / 3
    return np.sqrt(eval_0a + eval_ab + eval_b1)


def true_g_ux_H1_seminorm(ax, bx):
    mx = 0.5 * (ax + bx)
    dx = bx - ax
    return np.sqrt(mx * (1 - mx) - dx / 6)


def true_g_uy_H0_norm(ay, by):
    # same as ux norm due to symmetry
    ax, bx = ay, by
    return true_g_ux_H0_norm(ax, bx)


def true_g_uy_H1_seminorm(ay, by):
    # same as ux norm due to symmetry
    ax, bx = ay, by
    return true_g_ux_H1_seminorm(ax, bx)

In [ ]:
def true_gg_inner_product(a, b, *, norm):

    if dimensionality == 1:
        ax, bx = a[0], b[0]

        if norm == "energy":
            dux_dx_norm = true_g_ux_H1_seminorm(ax, bx)
            return dux_dx_norm**2
        elif norm == "l2":
            ux_norm = true_g_ux_H0_norm(ax, bx)
            return ux_norm**2
        else:
            assert False

    elif dimensionality == 2:
        ax, ay = a
        bx, by = b

        if norm == "energy":
            ux_norm = true_g_ux_H0_norm(ax, bx)
            uy_norm = true_g_uy_H0_norm(ay, by)
            dux_dx_norm = true_g_ux_H1_seminorm(ax, bx)
            duy_dy_norm = true_g_uy_H1_seminorm(ay, by)
            return dux_dx_norm**2 * uy_norm**2 + ux_norm**2 * duy_dy_norm**2

        elif norm == "l2":
            ux_norm = true_g_ux_H0_norm(ax, bx)
            uy_norm = true_g_uy_H0_norm(ay, by)
            return ux_norm**2 * uy_norm**2

        else:
            assert False

    else:
        assert False

In [ ]:
def fem_g_source(a, b, *, globdat):
    nodes = globdat["nodeSet"]
    elems = globdat["elemSet"]
    bboxes = globdat["bboxes"]
    dofs = globdat["dofSpace"]
    shape = globdat["shape"]

    assert nodes is elems.get_nodes()

    dof_types = dofs.get_types()
    g = np.zeros(dofs.dof_count())

    if dimensionality == 1:
        bbox_ab = (np.array([a[0]]), np.array([b[0]]))
        ielems = list_bbox_bbox_intersections(bboxes, bbox_ab)

        dx = b[0] - a[0]
        height = 1 / dx

        for ielem in ielems:
            inodes = elems[ielem]
            coords = nodes[inodes]
            idofs = dofs.get_dofs(inodes, dof_types)

            # get intersection
            line = np.array([[max(a[0], coords[0, 0])], [min(b[0], coords[1, 0])]])
            ipoint = np.mean(line, axis=0)
            area = np.abs(line[1] - line[0])
            loc_point = shape.get_local_point(ipoint, coords)
            sfuncs = shape.eval_shape_functions(loc_point)
            g[idofs] += area * height * sfuncs

        return g

    elif dimensionality == 2:
        dx = b[0] - a[0]
        dy = b[1] - a[1]

        bbox_ab_ab = (np.array([a[0], a[1]]), np.array([b[0], b[1]]))
        bbox_0a_ab = (np.array([0.0, a[1]]), np.array([a[0], b[1]]))
        bbox_b1_ab = (np.array([b[0], a[1]]), np.array([1.0, b[1]]))
        bbox_ab_0a = (np.array([a[0], 0.0]), np.array([b[0], a[1]]))
        bbox_ab_b1 = (np.array([a[0], b[1]]), np.array([b[0], 1.0]))

        poly_ab_ab = np.array([[a[0], a[1]], [b[0], a[1]], [b[0], b[1]], [a[0], b[1]]])
        poly_0a_ab = np.array([[0.0, a[1]], [a[0], a[1]], [a[0], b[1]], [0.0, b[1]]])
        poly_b1_ab = np.array([[b[0], a[1]], [1.0, a[1]], [1.0, b[1]], [b[0], b[1]]])
        poly_ab_0a = np.array([[a[0], 0.0], [b[0], 0.0], [b[0], a[1]], [a[0], a[1]]])
        poly_ab_b1 = np.array([[a[0], b[1]], [b[0], b[1]], [b[0], 1.0], [a[0], 1.0]])

        bbox_list = [bbox_ab_ab, bbox_0a_ab, bbox_b1_ab, bbox_ab_0a, bbox_ab_b1]
        poly_list = [poly_ab_ab, poly_0a_ab, poly_b1_ab, poly_ab_0a, poly_ab_b1]
        var_list = ["xy", "x", "x", "y", "y"]

        for i, (bbox, poly, var) in enumerate(zip(bbox_list, poly_list, var_list)):
            ielems = list_bbox_bbox_intersections(bboxes, bbox)

            for ielem in ielems:
                elem_bbox = (bboxes[0][ielem], bboxes[1][ielem])
                inodes = elems[ielem]
                coords = nodes[inodes]
                idofs = dofs.get_dofs(inodes, dof_types)

                # get convex polygon intersection
                if np.all(elem_bbox[0] > bbox[0]) and np.all(elem_bbox[1] < bbox[1]):
                    clip = coords
                else:
                    clip = clip_polygons(coords, poly)

                # skip element if no intersection is found
                if len(clip) == 0:
                    continue

                # fan triangulation
                for ic in range(0, len(clip) - 2):
                    tri = clip[[ic, ic + 1, -1]]
                    ipoint = np.mean(tri, axis=0)
                    area = 0.5 * np.linalg.det(np.hstack([tri, np.ones((3, 1))]))

                    loc_point = shape.get_local_point(ipoint, coords)
                    sfuncs = shape.eval_shape_functions(loc_point)

                    if "x" in var:
                        d2uy_dy2 = true_g_d2uy_dy2(ipoint[1], a[1], b[1])
                        ux = true_g_ux(ipoint[0], a[0], b[0])
                        assert d2uy_dy2 < 0.0
                        assert ux > 0.0
                        g[idofs] -= area * d2uy_dy2 * ux * sfuncs

                    if "y" in var:
                        d2ux_dx2 = true_g_d2ux_dx2(ipoint[0], a[0], b[0])
                        uy = true_g_uy(ipoint[1], a[1], b[1])
                        assert d2ux_dx2 < 0.0
                        assert uy > 0.0
                        g[idofs] -= area * d2ux_dx2 * uy * sfuncs

        return g

    else:
        assert False

In [ ]:
def fem_g_solution(a, b, *, globdat, g=None):
    nodes = globdat["nodeSet"]
    elems = globdat["elemSet"]
    dofs = globdat["dofSpace"]
    shape = globdat["shape"]

    assert nodes is elems.get_nodes()

    if g is None:
        g = fem_prior_source(a, b, globdat=globdat)

    Kc = globdat["Kc"]
    cdofs = globdat["constraints"].get_constraints()[0]
    g[cdofs] = 0.0

    ug = Kc.solve_A(g)
    return ug


def fem_gg_inner_product(a, b, *, norm, globdat):
    g = fem_g_source(a, b, globdat=globdat)
    ug = fem_g_solution(a, b, globdat=globdat, g=g)

    if norm == "energy":
        return ug @ g
    elif norm == "l2":
        M = globdat["matrix2"]
        return ug @ M @ ug
    else:
        assert False

In [ ]:
def fem_quadrature(func, *, mesh, dofs, shape, args={}):
    nodes, elems = mesh
    dof_types = dofs.get_types()
    type_count = len(dof_types)
    dof_count = shape.node_count() * type_count
    integral = np.zeros(dofs.dof_count())

    N = np.zeros((type_count, dof_count))
    b = np.zeros(type_count)
    elint = np.zeros(dof_count)

    warn("assuming all elements have exactly the same shape")
    sfuncs = shape.get_shape_functions()
    coords_0 = nodes[elems[0]]
    iwts = shape.get_integration_weights(coords_0)
    ip_count = len(iwts)
    Ns = np.zeros((ip_count, type_count, dof_count))

    for ip in range(len(iwts)):
        for i in range(type_count):
            N[i, i::type_count] = sfuncs[ip]
        Ns[ip] = N

    for ielem, inodes in enumerate(elems):
        coords = nodes[inodes]
        idofs = dofs.get_dofs(inodes, types=dof_types)
        ipoints = shape.get_global_integration_points(coords)

        elint[:] = 0.0

        for ip, (ipoint, iwt) in enumerate(zip(ipoints, iwts)):
            for i in range(type_count):
                # N[i, i::type_count] = sfuncs[ip]
                b[i] = func(ipoint, **args)

            elint += iwt * Ns[ip].T @ b
        integral[idofs] += elint

    return integral

In [ ]:
def true_prior_std(a, b, *, norm):
    return np.sqrt(true_gg_inner_product(a, b, norm=norm))


def fem_prior_std(a, b, *, norm, globdat_ref):
    return np.sqrt(fem_gg_inner_product(a, b, norm=norm, globdat=globdat_ref))


def true_posterior_std(a, b, *, norm, globdat_obs):
    std_prior = true_prior_std(a, b, norm=norm)
    var_downdate = fem_gg_inner_product(a, b, norm=norm, globdat=globdat_obs)
    return np.sqrt(std_prior**2 - var_downdate)


def fem_posterior_std(a, b, *, norm, globdat_obs, globdat_ref):
    std_prior_h = fem_prior_std(a, b, norm=norm, globdat_ref=globdat_ref)
    var_downdate = fem_gg_inner_product(a, b, norm=norm, globdat=globdat_obs)
    return np.sqrt(std_prior_h**2 - var_downdate)


def posterior_mean(a, b, *, norm, globdat_obs):
    obs_nodes = globdat_obs["nodeSet"]
    obs_elems = globdat_obs["elemSet"]

    g = fem_g_source(a, b, globdat=globdat_obs)

    if norm == "energy":
        Kc = globdat_obs["Kc"]
        fc = globdat_obs["extForce"].copy()
        cdofs = globdat_obs["constraints"].get_constraints()[0]

        fc[cdofs] = 0.0
        uf = Kc.solve_A(fc)
        return g @ uf

    elif norm == "l2":
        Mc = globdat_obs["Mc"]
        fc = globdat_obs["extForce"].copy()
        cdofs = globdat_obs["constraints"].get_constraints()[0]
        fc[cdofs] = 0.0
        fhat = Mc.solve_A(fc)

        dofs = globdat_obs["dofSpace"]
        shape = globdat_obs["shape"]

        ug = fem_quadrature(
            true_g_solution,
            args={"a": a, "b": b},
            mesh=obs_mesh,
            dofs=dofs,
            shape=shape,
        )

        return ug @ fhat
    else:
        assert False

In [ ]:
if dimensionality == 1:
    ns = 2 ** np.arange(2, 18)
elif dimensionality == 2:
    ns = 2 ** np.arange(1, 9)

meshes = []
globdats = []

for n in ns:
    if dimensionality == 1:
        mesh = mesh_interval_with_line2(n=n)
    elif dimensionality == 2:
        mesh = mesh_rectangle_with_tri3(n=n)

    nodes, elems = mesh
    props = get_fem_props(dimensionality)
    jive = CJiveRunner(props, elems=elems)
    globdat = jive()

    dofs = globdat["dofSpace"]
    shape = globdat["shape"]
    cdofs = globdat["constraints"].get_constraints()[0]

    Kc = globdat["matrix0"].copy()
    Kc[:, cdofs] *= 0.0
    Kc[cdofs, :] *= 0.0
    Kc[cdofs, cdofs] = 1.0
    globdat["Kc"] = cholesky(csc_matrix(Kc))

    M = create_unit_mass_matrix(elems, dofs, shape, sparse=True, lumped=False)
    Mc = M.copy()
    Mc[:, cdofs] *= 0.0
    Mc[cdofs, :] *= 0.0
    Mc[cdofs, cdofs] = 1.0
    globdat["matrix2"] = M
    globdat["Mc"] = cholesky(csc_matrix(Mc))

    globdat["bboxes"] = create_bboxes(elems)

    meshes.append(mesh)
    globdats.append(globdat)

In [ ]:
d = 0.01
norm = "energy"

if dimensionality == 1:
    mxs = np.linspace(0, 1, 15, endpoint=False)
    mxs += 0.5 * (mxs[1] - mxs[0])
    ms = mxs.reshape(-1, 1)
elif dimensionality == 2:
    mxs = np.linspace(0, 1, 5, endpoint=False)
    mxs += 0.5 * (mxs[1] - mxs[0])
    mys = np.linspace(0, 1, 5, endpoint=False)
    mys += 0.5 * (mys[1] - mys[0])
    mXs, mYs = np.meshgrid(mxs, mys)
    ms = np.column_stack([mXs.ravel(), mYs.ravel()])

prior_ref_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    ref_mesh = meshes[i]
    globdat_ref = globdats[i]

    for j, m in enumerate(ms):
        print(n, m)
        a = m - 0.5 * d
        b = m + 0.5 * d
        sigma_prior = true_prior_std(a, b, norm=norm)
        sigma_prior_h = fem_prior_std(a, b, norm=norm, globdat_ref=globdat_ref)
        prior_ref_W2_distances[i, j] = abs(sigma_prior - sigma_prior_h)

In [ ]:
fig, ax = plt.subplots()
ax.loglog(ns, prior_ref_W2_distances)
ax.set_aspect("equal")
plt.grid()
plt.show()

### 1D posterior convergence with reference mesh refinement

In [ ]:
if dimensionality == 1:
    obs_mesh = mesh_interval_with_line2(n=4)
elif dimensionality == 2:
    obs_mesh = mesh_rectangle_with_tri3(n=2)

obs_nodes, obs_elems = obs_mesh
props = get_fem_props(dimensionality)
jive = CJiveRunner(props, elems=obs_elems)
globdat_obs = jive()

dofs = globdat_obs["dofSpace"]
shape = globdat_obs["shape"]
cdofs = globdat_obs["constraints"].get_constraints()[0]

Kc = globdat_obs["matrix0"].copy()
Kc[:, cdofs] *= 0.0
Kc[cdofs, :] *= 0.0
Kc[cdofs, cdofs] = 1.0
globdat_obs["Kc"] = cholesky(csc_matrix(Kc))

M_obs = create_unit_mass_matrix(obs_elems, dofs, shape, sparse=True, lumped=False)
globdat_obs["matrix2"] = M_obs
Mc = M_obs.copy()
Mc[:, cdofs] *= 0.0
Mc[cdofs, :] *= 0.0
Mc[cdofs, cdofs] = 1.0
globdat_obs["Mc"] = cholesky(csc_matrix(Mc))


globdat_obs["bboxes"] = create_bboxes(obs_elems)

post_ref_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    ref_mesh = meshes[i]
    globdat_ref = globdats[i]

    for j, m in enumerate(ms):
        print(n, m)
        a = m - 0.5 * d
        b = m + 0.5 * d
        sigma_post = true_posterior_std(a, b, norm=norm, globdat_obs=globdat_obs)
        sigma_post_h = fem_posterior_std(
            a, b, norm=norm, globdat_obs=globdat_obs, globdat_ref=globdat_ref
        )
        post_ref_W2_distances[i, j] = abs(sigma_post - sigma_post_h)

In [ ]:
fig, ax = plt.subplots()
ax.loglog(ns, post_ref_W2_distances)
ax.set_aspect("equal")
plt.grid()
plt.show()

### 1D posterior convergence with observation mesh refinement

In [ ]:
def true_integral_ugx_ugx(ax, bx):
    mx = 0.5 * (ax + bx)
    dx = bx - ax
    A = 0.5 - mx
    B = mx * (1 - mx) - dx / 8

    # int_0^a -> 1/3 (1 - m)**2 a**3
    # int_b^1 -> 1/3 m**2 (1 - b)**3
    # int_a^b -> 1/320 d**3 + 1/12 A**2 d**3 - 1/12 B d**2 + B**2 d
    eval_0a = (1 - mx) ** 2 * ax**3 / 3
    eval_ab = dx**3 / 320 + A**2 * dx**3 / 12 - B * dx**2 / 12 + B**2 * dx
    eval_b1 = mx**2 * (1 - bx) ** 3 / 3

    # from scipy.integrate import quad

    # def integrand(x):
    #     return true_g_ux(x, ax, bx) * true_g_ux(x, ax, bx)

    # num_approx = quad(integrand, 0, 1)[0]

    # print(num_approx, eval_0a + eval_ab + eval_b1)

    return eval_0a + eval_ab + eval_b1


def true_integral_ux_ugx(ax, bx):
    mx = 0.5 * (ax + bx)
    dx = bx - ax

    def I1(x):
        # I1 = int x * (1 - mx) * (1 - x**2) * (exp(2 * x) - 1) dx
        #    = (mx - 1) / 8 * (-2 * x**4 + 4 * x**2 + exp(2 * x) * (4 * x**3 - 6 * x**2 + 2 * x - 1))
        return (
            0.125
            * (mx - 1)
            * (-2 * x**4 + 4 * x**2 + np.exp(2 * x) * (4 * x**3 - 6 * x**2 + 2 * x - 1))
        )

    def I2(x):
        # I2 = int ((x - ax) * (x - bx) / (2 * dx) + (0.5 - mx) * x + 0.5 * ax) * (1 - x**2) * (exp(2 * x) - 1) dx
        #    = int (-x**2 / (2 * dx) + (mx / dx + 0.5 - mx) * x + 0.5 * ax - ax * bx / (2 * dx)) * (1 - x**2) * (exp(2 * x) - 1) dx
        #    = int (alpha x**2 + beta x + gamma) * (1 - x**2) * (exp(2 * x) - 1) dx
        #    = alpha x**5 / 5 + beta x**4 / 4 + x**3 (gamma - alpha) / 3 - alpha / 2 * x**2
        alpha = -0.5 / dx
        beta = mx / dx + 0.5 - mx
        gamma = 0.5 * ax - (ax * bx) / (2 * dx)

        exp_part = (
            0.125
            * np.exp(2 * x)
            * (
                beta
                + 2 * gamma
                - 4 * beta * x**3
                - 4 * alpha * (x - 1) ** 2 * (x**2 + 1)
                + 6 * beta * x**2
                - 4 * gamma * x**2
                - 2 * beta * x
                + 4 * gamma * x
            )
        )
        pol_part = (
            x
            / 60
            * (
                -60 * gamma
                + 12 * alpha * x**4
                + 15 * beta * x**3
                - 20 * x**2 * (alpha - gamma)
                - 30 * beta * x
            )
        )

        return exp_part + pol_part

    def I3(x):
        # I3 = int (1 - x) * mx * (1 - x**2) * (exp(2 * x) - 1) dx
        #    = m / 24 * (-2 * (x - 1)**3 * (3 * x + 5) + 3 * exp(2 * x) * (4 * x**3 - 10 * x**2 + 6 * x + 1))
        return (
            mx
            / 24
            * (
                3 * np.exp(2 * x) * (4 * x**3 - 10 * x**2 + 6 * x + 1)
                - 2 * (x - 1) ** 3 * (3 * x + 5)
            )
        )

    # from scipy.integrate import quad

    # def integrand(x):
    #     return true_f_ux(x) * true_g_ux(x, ax, bx)

    # print(quad(integrand, 0.0, ax)[0], I1(ax) - I1(0.0))
    # print(quad(integrand, ax, bx)[0], I2(bx) - I2(ax))
    # print(quad(integrand, bx, 1.0)[0], I3(1.0) - I3(bx))

    # num_approx = quad(integrand, 0, 1)[0]
    # print(num_approx, I1(ax) - I1(0.0) + I2(bx) - I2(ax) + I3(1.0) - I3(bx))

    return I1(ax) - I1(0.0) + I2(bx) - I2(ax) + I3(1.0) - I3(bx)


def true_integral_uy_ugy(ay, by):
    my = 0.5 * (ay + by)
    dy = by - ay

    def I1(y):
        # I1 = int y * (1 - my) * sin(pi * y) dy
        #    = (1 - my) * (sin(pi * y) / pi**2 - y * cos(pi * y) / pi)
        return (1 - my) * (np.sin(np.pi * y) / np.pi**2 - y * np.cos(np.pi * y) / np.pi)

    def I2(y):
        # I2 = int ((y - ay) * (y - by) / (2 * dy) + (0.5 - my) * y + 0.5 * ay) * sin(pi * y) dy
        #    = int (-y**2 / (2 * dy) + (my / dy + 0.5 - my) * y + 0.5 * ay - ay * by / (2 * dy)) * sin(pi * y) dy
        #    = int (alpha * y**2 + beta * y + gamma) * sin(pi * y) dy
        #    = (y^2 / (2 * dy * pi) - alpha * y / pi - beta / pi - 1 / (dy * pi**3)) * cos(pi * x)
        #      + (-x / (dy * pi**2) + alpha / pi**2) * sin(pi * x)
        alpha = -0.5 / dy
        beta = my / dy + 0.5 - my
        gamma = 0.5 * ay - (ay * by) / (2 * dy)
        cos_part = (
            -np.cos(np.pi * y)
            / np.pi**3
            * (
                -2 * alpha
                + np.pi**2 * alpha * y**2
                + np.pi**2 * beta * y
                + np.pi**2 * gamma
            )
        )
        sin_part = np.sin(np.pi * y) / np.pi**2 * (beta + 2 * alpha * y)

        return cos_part + sin_part

    def I3(y):
        # I3 = int (1 - y) * my * sin(pi * y) dy
        #    = -my * (sin(pi * y) / pi**2 + (1 - y) * cos(pi * y) / pi)
        return -my * (
            np.sin(np.pi * y) / np.pi**2 + (1 - y) * np.cos(np.pi * y) / np.pi
        )

    # from scipy.integrate import quad

    # def integrand(y):
    #     return true_f_uy(y) * true_g_uy(y, ay, by)

    # print(quad(integrand, 0.0, ay)[0], I1(ay) - I1(0.0))
    # print(quad(integrand, ay, by)[0], I2(by) - I2(ay))
    # print(quad(integrand, by, 1.0)[0], I3(1.0) - I3(by))

    # num_approx = quad(integrand, 0, 1)[0]
    # print(num_approx, I1(ay) - I1(0.0) + I2(by) - I2(ay) + I3(1.0) - I3(by))

    return I1(ay) - I1(0.0) + I2(by) - I2(ay) + I3(1.0) - I3(by)


def true_integral_dux_dx_dugx_dx(ax, bx):
    dx = bx - ax

    def I(x):
        # I = int (1 - x**2) * (exp(2*x) - 1) dx
        #   = exp(2*x) * (1/4 + x/2 - x**2/2) + (x**3/3 - x)
        exp_part = np.exp(2 * x) * (0.25 + 0.5 * x - 0.5 * x**2)
        pol_part = x**3 / 3 - x
        return (exp_part + pol_part) / dx

    # from scipy.integrate import quad

    # def integrand(x):
    #     return true_f_dux_dx(x) * true_g_dux_dx(x, ax, bx)

    # print(quad(integrand, 0.0, 1.0)[0], I(bx) - I(ax))

    return I(bx) - I(ax)


def true_integral_duy_dy_dugy_dy(ay, by):
    dy = by - ay

    def I(y):
        # I = int 1 / dy * sin(pi * y) dy
        #   = - 1 / (dy * pi) * cos(pi * y)
        return -np.cos(np.pi * y) / (dy * np.pi)

    # from scipy.integrate import quad

    # def integrand(y):
    #     return true_f_duy_dy(y) * true_g_duy_dy(y, ay, by)

    # print(quad(integrand, 0.0, 1.0)[0], I(by) - I(ay))

    return I(by) - I(ay)


def true_integral_dugx_dx_dugx_dx(ax, bx):
    mx = 0.5 * (ax + bx)
    dx = bx - ax
    return mx * (1 - mx) - dx / 6


def true_g_uy_H0_norm(ay, by):
    # same as ux norm due to symmetry
    ax, bx = ay, by
    return true_g_ux_H0_norm(ax, bx)


def true_g_uy_H1_seminorm(ay, by):
    # same as ux norm due to symmetry
    ax, bx = ay, by
    return true_g_ux_H1_seminorm(ax, bx)

In [ ]:
def true_qoi(a, b):

    if dimensionality == 1:
        # (1 - x**2) * (np.exp(2 * x) - 1)
        # -1            -> -x
        # x^2           -> 1/3 x^3
        # exp(2x)       -> 1/2 exp(2x)
        # -x^2 exp(2x)  -> (-1/2 x^2 + 1/2 x - 1/4) exp(2x)

        # total -> 1/3 x^3 - x + (1/4 + 1/2 x - 1/2 x^2) exp(2x)
        eval_a = a**3 / 3 - a + (0.25 + 0.5 * a - 0.5 * a**2) * np.exp(2 * a)
        eval_b = b**3 / 3 - b + (0.25 + 0.5 * b - 0.5 * b**2) * np.exp(2 * b)
        return (eval_b - eval_a) / (b - a)

    elif dimensionality == 2:
        # <u, g> = int_0^1 int_0^1 u_x * u_y * (d^2ug_x/dx^2 * ug_y + ug_x * d^2ug_y/dy^2) dx dy
        #        = int_0^1 u_x ug_x dx * int_0^1 du_y/dy dug_y/dy dy + int_0^1 du_x/dx dug_x/dx dx * int_0^1 u_y ug_y dy
        #        = I1                  * I2                          + I3                          * I4
        ax, ay = a
        bx, by = b

        I1 = true_integral_ux_ugx(ax, bx)
        I2 = true_integral_duy_dy_dugy_dy(ay, by)
        I3 = true_integral_dux_dx_dugx_dx(ax, bx)
        I4 = true_integral_uy_ugy(ay, by)

        I = I1 * I2 + I3 * I4

        # from scipy.integrate import dblquad

        # def integrand(y, x):
        #     coord = np.array([x, y])
        #     return true_f_solution_2d(coord) * true_g_source_2d(coord, a, b)

        # num_approx = (
        #     dblquad(integrand, ax, bx, ay, by)[0]
        #     + dblquad(integrand, ax, bx, 0.0, ay)[0]
        #     + dblquad(integrand, ax, bx, by, 1.0)[0]
        #     + dblquad(integrand, 0.0, ax, ay, by)[0]
        #     + dblquad(integrand, bx, 1.0, ay, by)[0]
        # )

        # print(I, num_approx)
        # assert False
        return I

In [ ]:
post_obs_W2_distances = np.zeros((len(ns), len(ms)))

for i, n in enumerate(ns):
    obs_mesh = meshes[i]
    globdat_obs = globdats[i]

    for j, m in enumerate(ms):
        print(n, m)
        a = m - 0.5 * d
        b = m + 0.5 * d
        mu_post = posterior_mean(a, b, norm=norm, globdat_obs=globdat_obs)
        mu_true = true_qoi(a, b)
        sigma_post = true_posterior_std(a, b, norm=norm, globdat_obs=globdat_obs)
        post_obs_W2_distances[i, j] = np.sqrt((mu_true - mu_post) ** 2 + sigma_post**2)

In [ ]:
fig, ax = plt.subplots()
ax.loglog(ns, post_obs_W2_distances)
ax.set_aspect("equal")
plt.grid()
plt.show()